In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# 【读取步骤】
# 读取 POS_CASH_balance 数据
# ============================================================


# 1. 读取原始数据
pos_cash = pd.read_csv(
    "data/raw/home_credit/POS_CASH_balance.csv"
)


# 2. 查看数据规模
print(
    "POS_CASH_balance shape:",
    pos_cash.shape
)


# 3. 查看前几行
display(
    pos_cash.head()
)


# 4. 查看字段类型
pos_cash.info()

In [ ]:
# ============================================================
# 【检查步骤 1】
# 检查数据规模和粒度
# ============================================================


print(
    "总月度记录数量:",
    pos_cash.shape[0]
)

print(
    "唯一客户数量:",
    pos_cash["SK_ID_CURR"].nunique()
)

print(
    "唯一历史贷款数量:",
    pos_cash["SK_ID_PREV"].nunique()
)

In [ ]:
# ============================================================
# 【检查步骤 2】
# 检查重复值
# ============================================================


print(
    "完全重复行数量:",
    pos_cash.duplicated().sum()
)


print(
    "贷款月份组合重复数量:",
    pos_cash.duplicated(
        subset=[
            "SK_ID_PREV",
            "MONTHS_BALANCE"
        ]
    ).sum()
)

In [ ]:
# ============================================================
# 【检查步骤 3】
# 检查缺失值
# ============================================================


pos_cash_missing = (
    pos_cash
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


pos_cash_missing[
    "missing_rate"
] = (
    pos_cash_missing[
        "missing_count"
    ]
    / len(pos_cash)
)


pos_cash_missing

In [ ]:
# ============================================================
# 【特征工程 1】
# 建立 POS_CASH 月度逾期与贷款状态辅助特征
# ============================================================


# 1. 标记该月是否发生逾期
pos_cash["IS_POS_DPD"] = (
    pos_cash["SK_DPD"] > 0
).astype("int8")


# 2. 标记该月是否发生考虑容忍规则后的逾期
pos_cash["IS_POS_DPD_DEF"] = (
    pos_cash["SK_DPD_DEF"] > 0
).astype("int8")


# 3. 标记该月是否发生严重逾期
pos_cash["IS_POS_SEVERE_DPD"] = (
    pos_cash["SK_DPD"] >= 30
).astype("int8")


# 4. 标记该月是否发生严重的容忍后逾期
pos_cash["IS_POS_SEVERE_DPD_DEF"] = (
    pos_cash["SK_DPD_DEF"] >= 30
).astype("int8")


# 5. 标记贷款是否处于活跃状态
pos_cash["IS_POS_ACTIVE"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Active"
).astype("int8")


# 6. 标记贷款是否已经完成
pos_cash["IS_POS_COMPLETED"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Completed"
).astype("int8")


# 7. 标记是否发生合同要求状态
pos_cash["IS_POS_DEMAND"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Demand"
).astype("int8")


# 8. 标记是否发生合同取消
pos_cash["IS_POS_CANCELED"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Canceled"
).astype("int8")


# 9. 标记是否存在其他非正常状态
pos_cash["IS_POS_OTHER_STATUS"] = (
    ~pos_cash["NAME_CONTRACT_STATUS"].isin(
        [
            "Active",
            "Completed",
            "Demand",
            "Canceled"
        ]
    )
).astype("int8")


# 10. 建立贷款已完成期数
pos_cash["POS_INSTALMENT_PAID_COUNT"] = (
    pos_cash["CNT_INSTALMENT"]
    - pos_cash["CNT_INSTALMENT_FUTURE"]
)


# 11. 建立贷款完成进度比例
pos_cash["POS_INSTALMENT_PROGRESS_RATIO"] = (
    pos_cash["POS_INSTALMENT_PAID_COUNT"]
    / pos_cash["CNT_INSTALMENT"].replace(0, np.nan)
)


# 12. 查看结果
pos_cash[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "MONTHS_BALANCE",
        "CNT_INSTALMENT",
        "CNT_INSTALMENT_FUTURE",
        "POS_INSTALMENT_PAID_COUNT",
        "POS_INSTALMENT_PROGRESS_RATIO",
        "NAME_CONTRACT_STATUS",
        "SK_DPD",
        "SK_DPD_DEF",
        "IS_POS_DPD",
        "IS_POS_DPD_DEF",
        "IS_POS_SEVERE_DPD",
        "IS_POS_SEVERE_DPD_DEF",
        "IS_POS_ACTIVE",
        "IS_POS_COMPLETED",
        "IS_POS_DEMAND",
        "IS_POS_CANCELED",
        "IS_POS_OTHER_STATUS"
    ]
].head(10)

In [ ]:
# 1. 查看贷款状态分布
pos_cash[
    "NAME_CONTRACT_STATUS"
].value_counts(
    dropna=False
)

In [ ]:
# 1. 查看逾期天数分布
pos_cash[
    [
        "SK_DPD",
        "SK_DPD_DEF"
    ]
].describe().T


# 2. 查看逾期记录数量
print(
    "发生原始逾期的月度记录数量:",
    pos_cash["IS_POS_DPD"].sum()
)

print(
    "发生容忍后逾期的月度记录数量:",
    pos_cash["IS_POS_DPD_DEF"].sum()
)

print(
    "发生30天及以上逾期的记录数量:",
    pos_cash["IS_POS_SEVERE_DPD"].sum()
)

In [ ]:
pos_cash[
    [
        "CNT_INSTALMENT",
        "CNT_INSTALMENT_FUTURE",
        "POS_INSTALMENT_PAID_COUNT",
        "POS_INSTALMENT_PROGRESS_RATIO"
    ]
].describe().T

In [ ]:
print(
    "完成期数小于0的记录数量:",
    (
        pos_cash["POS_INSTALMENT_PAID_COUNT"] < 0
    ).sum()
)

print(
    "进度比例小于0的记录数量:",
    (
        pos_cash["POS_INSTALMENT_PROGRESS_RATIO"] < 0
    ).sum()
)

print(
    "进度比例大于1的记录数量:",
    (
        pos_cash["POS_INSTALMENT_PROGRESS_RATIO"] > 1
    ).sum()
)

In [ ]:
# ============================================================
# 【特征工程 2】
# 建立贷款级 POS_CASH 历史表现特征
# ============================================================


# 1. 找出每笔贷款距离当前申请日最近的一条月度记录
pos_latest_month = (
    pos_cash
    .sort_values(
        [
            "SK_ID_PREV",
            "MONTHS_BALANCE"
        ]
    )
    .groupby("SK_ID_PREV")
    .tail(1)
    [
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
            "MONTHS_BALANCE",
            "CNT_INSTALMENT",
            "CNT_INSTALMENT_FUTURE",
            "POS_INSTALMENT_PAID_COUNT",
            "POS_INSTALMENT_PROGRESS_RATIO",
            "NAME_CONTRACT_STATUS",
            "IS_POS_ACTIVE",
            "IS_POS_COMPLETED",
            "IS_POS_DEMAND",
            "IS_POS_CANCELED",
            "IS_POS_OTHER_STATUS"
        ]
    ]
    .rename(
        columns={
            "MONTHS_BALANCE": "POS_LATEST_MONTH",
            "CNT_INSTALMENT": "POS_LATEST_INSTALMENT_COUNT",
            "CNT_INSTALMENT_FUTURE": "POS_LATEST_FUTURE_INSTALMENT_COUNT",
            "POS_INSTALMENT_PAID_COUNT": "POS_LATEST_PAID_INSTALMENT_COUNT",
            "POS_INSTALMENT_PROGRESS_RATIO": "POS_LATEST_PROGRESS_RATIO",
            "NAME_CONTRACT_STATUS": "POS_LATEST_CONTRACT_STATUS",
            "IS_POS_ACTIVE": "POS_LATEST_IS_ACTIVE",
            "IS_POS_COMPLETED": "POS_LATEST_IS_COMPLETED",
            "IS_POS_DEMAND": "POS_LATEST_IS_DEMAND",
            "IS_POS_CANCELED": "POS_LATEST_IS_CANCELED",
            "IS_POS_OTHER_STATUS": "POS_LATEST_IS_OTHER_STATUS"
        }
    )
)


# 2. 按历史贷款聚合全部月度表现
pos_loan_history_features = (
    pos_cash
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV"
        ]
    )
    .agg(
        # 该笔贷款包含的历史月份数量
        POS_HISTORY_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        # 最早出现月份
        POS_EARLIEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        # 最近出现月份
        POS_MOST_RECENT_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        # 原始逾期月份数量
        POS_DPD_MONTH_COUNT=(
            "IS_POS_DPD",
            "sum"
        ),

        # 容忍后逾期月份数量
        POS_DPD_DEF_MONTH_COUNT=(
            "IS_POS_DPD_DEF",
            "sum"
        ),

        # 30天及以上严重逾期月份数量
        POS_SEVERE_DPD_MONTH_COUNT=(
            "IS_POS_SEVERE_DPD",
            "sum"
        ),

        # 容忍后30天及以上严重逾期月份数量
        POS_SEVERE_DPD_DEF_MONTH_COUNT=(
            "IS_POS_SEVERE_DPD_DEF",
            "sum"
        ),

        # 平均逾期天数
        POS_AVG_DPD=(
            "SK_DPD",
            "mean"
        ),

        # 最大逾期天数
        POS_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        # 平均容忍后逾期天数
        POS_AVG_DPD_DEF=(
            "SK_DPD_DEF",
            "mean"
        ),

        # 最大容忍后逾期天数
        POS_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        )
    )
    .reset_index()
)


# 3. 建立贷款级逾期月份比例
pos_loan_history_features[
    "POS_DPD_MONTH_RATE"
] = (
    pos_loan_history_features[
        "POS_DPD_MONTH_COUNT"
    ]
    / pos_loan_history_features[
        "POS_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 4. 建立贷款级容忍后逾期月份比例
pos_loan_history_features[
    "POS_DPD_DEF_MONTH_RATE"
] = (
    pos_loan_history_features[
        "POS_DPD_DEF_MONTH_COUNT"
    ]
    / pos_loan_history_features[
        "POS_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立贷款级严重逾期月份比例
pos_loan_history_features[
    "POS_SEVERE_DPD_MONTH_RATE"
] = (
    pos_loan_history_features[
        "POS_SEVERE_DPD_MONTH_COUNT"
    ]
    / pos_loan_history_features[
        "POS_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 合并每笔贷款的最近状态
pos_loan_features = (
    pos_loan_history_features
    .merge(
        pos_latest_month,
        on=[
            "SK_ID_CURR",
            "SK_ID_PREV"
        ],
        how="left",
        validate="one_to_one"
    )
)


# 7. 建立贷款历史覆盖时长
pos_loan_features[
    "POS_HISTORY_SPAN_MONTHS"
] = (
    pos_loan_features[
        "POS_MOST_RECENT_MONTH"
    ]
    - pos_loan_features[
        "POS_EARLIEST_MONTH"
    ]
)


# 8. 查看贷款级结果
pos_loan_features.head()

In [ ]:
# 1. 查看贷款级数据规模
print(
    "pos_loan_features shape:",
    pos_loan_features.shape
)


# 2. 检查唯一贷款数量
print(
    "唯一历史贷款数量:",
    pos_loan_features[
        "SK_ID_PREV"
    ].nunique()
)


# 3. 检查重复贷款数量
print(
    "重复历史贷款数量:",
    pos_loan_features[
        "SK_ID_PREV"
    ].duplicated().sum()
)

In [ ]:
pos_loan_features[
    [
        "POS_DPD_MONTH_RATE",
        "POS_DPD_DEF_MONTH_RATE",
        "POS_SEVERE_DPD_MONTH_RATE",
        "POS_LATEST_PROGRESS_RATIO"
    ]
].describe().T

In [ ]:
pos_loan_features[
    [
        "POS_HISTORY_MONTH_COUNT",
        "POS_HISTORY_SPAN_MONTHS",
        "POS_AVG_DPD",
        "POS_MAX_DPD",
        "POS_AVG_DPD_DEF",
        "POS_MAX_DPD_DEF"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 3】
# 建立客户级 POS_CASH 历史表现特征
# ============================================================


# 1. 按客户聚合历史贷款表现
pos_cash_customer_features = (
    pos_loan_features
    .groupby("SK_ID_CURR")
    .agg(
        # 客户历史 POS/CASH 贷款数量
        POS_LOAN_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        # 客户累计拥有的月度记录数量
        POS_TOTAL_HISTORY_MONTH_COUNT=(
            "POS_HISTORY_MONTH_COUNT",
            "sum"
        ),

        # 客户单笔贷款平均历史月份数量
        POS_AVG_HISTORY_MONTH_COUNT=(
            "POS_HISTORY_MONTH_COUNT",
            "mean"
        ),

        # 客户最长的一笔贷款历史月份数量
        POS_MAX_HISTORY_MONTH_COUNT=(
            "POS_HISTORY_MONTH_COUNT",
            "max"
        ),

        # 客户最早一笔 POS/CASH 记录所在月份
        POS_CUSTOMER_EARLIEST_MONTH=(
            "POS_EARLIEST_MONTH",
            "min"
        ),

        # 客户最近一笔 POS/CASH 记录所在月份
        POS_CUSTOMER_MOST_RECENT_MONTH=(
            "POS_MOST_RECENT_MONTH",
            "max"
        ),

        # 客户全部贷款累计逾期月份数量
        POS_TOTAL_DPD_MONTH_COUNT=(
            "POS_DPD_MONTH_COUNT",
            "sum"
        ),

        # 客户全部贷款累计容忍后逾期月份数量
        POS_TOTAL_DPD_DEF_MONTH_COUNT=(
            "POS_DPD_DEF_MONTH_COUNT",
            "sum"
        ),

        # 客户全部贷款累计严重逾期月份数量
        POS_TOTAL_SEVERE_DPD_MONTH_COUNT=(
            "POS_SEVERE_DPD_MONTH_COUNT",
            "sum"
        ),

        # 客户全部贷款累计容忍后严重逾期月份数量
        POS_TOTAL_SEVERE_DPD_DEF_MONTH_COUNT=(
            "POS_SEVERE_DPD_DEF_MONTH_COUNT",
            "sum"
        ),

        # 客户多笔贷款的平均逾期天数
        POS_AVG_LOAN_DPD=(
            "POS_AVG_DPD",
            "mean"
        ),

        # 客户历史最大逾期天数
        POS_MAX_DPD=(
            "POS_MAX_DPD",
            "max"
        ),

        # 客户多笔贷款的平均容忍后逾期天数
        POS_AVG_LOAN_DPD_DEF=(
            "POS_AVG_DPD_DEF",
            "mean"
        ),

        # 客户历史最大容忍后逾期天数
        POS_MAX_DPD_DEF=(
            "POS_MAX_DPD_DEF",
            "max"
        ),

        # 最近仍处于活跃状态的贷款数量
        POS_ACTIVE_LOAN_COUNT=(
            "POS_LATEST_IS_ACTIVE",
            "sum"
        ),

        # 最近已经完成的贷款数量
        POS_COMPLETED_LOAN_COUNT=(
            "POS_LATEST_IS_COMPLETED",
            "sum"
        ),

        # 最近处于 Demand 状态的贷款数量
        POS_DEMAND_LOAN_COUNT=(
            "POS_LATEST_IS_DEMAND",
            "sum"
        ),

        # 最近已取消的贷款数量
        POS_CANCELED_LOAN_COUNT=(
            "POS_LATEST_IS_CANCELED",
            "sum"
        ),

        # 最近处于其他状态的贷款数量
        POS_OTHER_STATUS_LOAN_COUNT=(
            "POS_LATEST_IS_OTHER_STATUS",
            "sum"
        ),

        # 客户多笔贷款的平均最新完成进度
        POS_AVG_LATEST_PROGRESS_RATIO=(
            "POS_LATEST_PROGRESS_RATIO",
            "mean"
        ),

        # 客户贷款中最高的最新完成进度
        POS_MAX_LATEST_PROGRESS_RATIO=(
            "POS_LATEST_PROGRESS_RATIO",
            "max"
        ),

        # 客户当前累计剩余期数
        POS_TOTAL_FUTURE_INSTALMENT_COUNT=(
            "POS_LATEST_FUTURE_INSTALMENT_COUNT",
            "sum"
        ),

        # 客户平均每笔贷款剩余期数
        POS_AVG_FUTURE_INSTALMENT_COUNT=(
            "POS_LATEST_FUTURE_INSTALMENT_COUNT",
            "mean"
        )
    )
    .reset_index()
)


# 2. 建立客户整体原始逾期月份比例
pos_cash_customer_features[
    "POS_TOTAL_DPD_MONTH_RATE"
] = (
    pos_cash_customer_features[
        "POS_TOTAL_DPD_MONTH_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 3. 建立客户整体容忍后逾期月份比例
pos_cash_customer_features[
    "POS_TOTAL_DPD_DEF_MONTH_RATE"
] = (
    pos_cash_customer_features[
        "POS_TOTAL_DPD_DEF_MONTH_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 4. 建立客户整体严重逾期月份比例
pos_cash_customer_features[
    "POS_TOTAL_SEVERE_DPD_MONTH_RATE"
] = (
    pos_cash_customer_features[
        "POS_TOTAL_SEVERE_DPD_MONTH_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立客户整体容忍后严重逾期月份比例
pos_cash_customer_features[
    "POS_TOTAL_SEVERE_DPD_DEF_MONTH_RATE"
] = (
    pos_cash_customer_features[
        "POS_TOTAL_SEVERE_DPD_DEF_MONTH_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_TOTAL_HISTORY_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立活跃贷款比例
pos_cash_customer_features[
    "POS_ACTIVE_LOAN_RATE"
] = (
    pos_cash_customer_features[
        "POS_ACTIVE_LOAN_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_LOAN_COUNT"
    ].replace(0, np.nan)
)


# 7. 建立已完成贷款比例
pos_cash_customer_features[
    "POS_COMPLETED_LOAN_RATE"
] = (
    pos_cash_customer_features[
        "POS_COMPLETED_LOAN_COUNT"
    ]
    / pos_cash_customer_features[
        "POS_LOAN_COUNT"
    ].replace(0, np.nan)
)


# 8. 建立客户 POS/CASH 历史跨度
pos_cash_customer_features[
    "POS_CUSTOMER_HISTORY_SPAN_MONTHS"
] = (
    pos_cash_customer_features[
        "POS_CUSTOMER_MOST_RECENT_MONTH"
    ]
    - pos_cash_customer_features[
        "POS_CUSTOMER_EARLIEST_MONTH"
    ]
)


# 9. 查看客户级结果
pos_cash_customer_features.head()

In [ ]:
# 1. 查看客户级特征表规模
print(
    "pos_cash_customer_features shape:",
    pos_cash_customer_features.shape
)


# 2. 查看唯一客户数量
print(
    "唯一客户数量:",
    pos_cash_customer_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    pos_cash_customer_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
pos_cash_customer_features[
    [
        "POS_TOTAL_DPD_MONTH_RATE",
        "POS_TOTAL_DPD_DEF_MONTH_RATE",
        "POS_TOTAL_SEVERE_DPD_MONTH_RATE",
        "POS_TOTAL_SEVERE_DPD_DEF_MONTH_RATE",
        "POS_ACTIVE_LOAN_RATE",
        "POS_COMPLETED_LOAN_RATE"
    ]
].describe().T

In [ ]:
pos_cash_customer_features[
    [
        "POS_LOAN_COUNT",
        "POS_TOTAL_HISTORY_MONTH_COUNT",
        "POS_TOTAL_DPD_MONTH_COUNT",
        "POS_TOTAL_SEVERE_DPD_MONTH_COUNT",
        "POS_ACTIVE_LOAN_COUNT",
        "POS_COMPLETED_LOAN_COUNT",
        "POS_TOTAL_FUTURE_INSTALMENT_COUNT",
        "POS_CUSTOMER_HISTORY_SPAN_MONTHS"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 4】
# 建立客户最近一年 POS_CASH 表现特征
# ============================================================


# 1. 筛选申请日前最近12个月的月度记录
recent_pos_cash = (
    pos_cash[
        pos_cash["MONTHS_BALANCE"] >= -12
    ]
    .copy()
)


# 2. 查看最近一年数据规模
print(
    "最近一年月度记录数量:",
    recent_pos_cash.shape[0]
)

print(
    "最近一年涉及客户数量:",
    recent_pos_cash["SK_ID_CURR"].nunique()
)


# 3. 按客户聚合最近一年表现
pos_cash_recent_features = (
    recent_pos_cash
    .groupby("SK_ID_CURR")
    .agg(
        # 最近一年月度记录数量
        POS_RECENT_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        # 最近一年涉及的历史贷款数量
        POS_RECENT_LOAN_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        # 最近一年原始逾期月份数量
        POS_RECENT_DPD_MONTH_COUNT=(
            "IS_POS_DPD",
            "sum"
        ),

        # 最近一年容忍后逾期月份数量
        POS_RECENT_DPD_DEF_MONTH_COUNT=(
            "IS_POS_DPD_DEF",
            "sum"
        ),

        # 最近一年严重逾期月份数量
        POS_RECENT_SEVERE_DPD_MONTH_COUNT=(
            "IS_POS_SEVERE_DPD",
            "sum"
        ),

        # 最近一年容忍后严重逾期月份数量
        POS_RECENT_SEVERE_DPD_DEF_MONTH_COUNT=(
            "IS_POS_SEVERE_DPD_DEF",
            "sum"
        ),

        # 最近一年平均原始逾期天数
        POS_RECENT_AVG_DPD=(
            "SK_DPD",
            "mean"
        ),

        # 最近一年最大原始逾期天数
        POS_RECENT_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        # 最近一年平均容忍后逾期天数
        POS_RECENT_AVG_DPD_DEF=(
            "SK_DPD_DEF",
            "mean"
        ),

        # 最近一年最大容忍后逾期天数
        POS_RECENT_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        ),

        # 最近一年活跃状态月份数量
        POS_RECENT_ACTIVE_MONTH_COUNT=(
            "IS_POS_ACTIVE",
            "sum"
        ),

        # 最近一年完成状态月份数量
        POS_RECENT_COMPLETED_MONTH_COUNT=(
            "IS_POS_COMPLETED",
            "sum"
        ),

        # 最近一年 Demand 状态月份数量
        POS_RECENT_DEMAND_MONTH_COUNT=(
            "IS_POS_DEMAND",
            "sum"
        ),

        # 最近一年平均贷款完成进度
        POS_RECENT_AVG_PROGRESS_RATIO=(
            "POS_INSTALMENT_PROGRESS_RATIO",
            "mean"
        ),

        # 最近一年平均剩余期数
        POS_RECENT_AVG_FUTURE_INSTALMENT_COUNT=(
            "CNT_INSTALMENT_FUTURE",
            "mean"
        ),

        # 最近一年最大剩余期数
        POS_RECENT_MAX_FUTURE_INSTALMENT_COUNT=(
            "CNT_INSTALMENT_FUTURE",
            "max"
        )
    )
    .reset_index()
)


# 4. 建立最近一年原始逾期月份比例
pos_cash_recent_features[
    "POS_RECENT_DPD_MONTH_RATE"
] = (
    pos_cash_recent_features[
        "POS_RECENT_DPD_MONTH_COUNT"
    ]
    / pos_cash_recent_features[
        "POS_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立最近一年容忍后逾期月份比例
pos_cash_recent_features[
    "POS_RECENT_DPD_DEF_MONTH_RATE"
] = (
    pos_cash_recent_features[
        "POS_RECENT_DPD_DEF_MONTH_COUNT"
    ]
    / pos_cash_recent_features[
        "POS_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立最近一年严重逾期月份比例
pos_cash_recent_features[
    "POS_RECENT_SEVERE_DPD_MONTH_RATE"
] = (
    pos_cash_recent_features[
        "POS_RECENT_SEVERE_DPD_MONTH_COUNT"
    ]
    / pos_cash_recent_features[
        "POS_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 7. 建立最近一年容忍后严重逾期月份比例
pos_cash_recent_features[
    "POS_RECENT_SEVERE_DPD_DEF_MONTH_RATE"
] = (
    pos_cash_recent_features[
        "POS_RECENT_SEVERE_DPD_DEF_MONTH_COUNT"
    ]
    / pos_cash_recent_features[
        "POS_RECENT_MONTH_COUNT"
    ].replace(0, np.nan)
)


# 8. 查看客户最近一年 POS_CASH 特征
pos_cash_recent_features.head()

In [ ]:
# 1. 查看近期客户级特征规模
print(
    "pos_cash_recent_features shape:",
    pos_cash_recent_features.shape
)


# 2. 检查唯一客户数量
print(
    "唯一客户数量:",
    pos_cash_recent_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    pos_cash_recent_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
pos_cash_recent_features[
    [
        "POS_RECENT_DPD_MONTH_RATE",
        "POS_RECENT_DPD_DEF_MONTH_RATE",
        "POS_RECENT_SEVERE_DPD_MONTH_RATE",
        "POS_RECENT_SEVERE_DPD_DEF_MONTH_RATE"
    ]
].describe().T

In [ ]:
pos_cash_recent_features[
    [
        "POS_RECENT_MONTH_COUNT",
        "POS_RECENT_LOAN_COUNT",
        "POS_RECENT_AVG_DPD",
        "POS_RECENT_MAX_DPD",
        "POS_RECENT_AVG_DPD_DEF",
        "POS_RECENT_MAX_DPD_DEF",
        "POS_RECENT_AVG_FUTURE_INSTALMENT_COUNT",
        "POS_RECENT_MAX_FUTURE_INSTALMENT_COUNT"
    ]
].describe().T

In [ ]:
# ============================================================
# 【合并步骤】
# 合并 POS_CASH 客户级整体特征与近期特征
# ============================================================


# 1. 以客户整体历史特征作为主表
pos_cash_features = (
    pos_cash_customer_features

    # 2. 合并客户最近一年特征
    .merge(
        pos_cash_recent_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)


# 3. 查看最终结果
pos_cash_features.head()

In [ ]:
# 1. 查看最终数据规模
print(
    "pos_cash_features shape:",
    pos_cash_features.shape
)


# 2. 检查唯一客户数量
print(
    "唯一客户数量:",
    pos_cash_features["SK_ID_CURR"].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    pos_cash_features["SK_ID_CURR"].duplicated().sum()
)


# 4. 与原始表唯一客户数量进行对比
print(
    "原始 POS_CASH 唯一客户数量:",
    pos_cash["SK_ID_CURR"].nunique()
)

In [ ]:
duplicate_suffix_columns = [
    col
    for col in pos_cash_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(
    "重复后缀字段:",
    duplicate_suffix_columns
)

In [ ]:
# 1. 统计缺失数量
pos_cash_missing_summary = (
    pos_cash_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


# 2. 计算缺失比例
pos_cash_missing_summary["missing_rate"] = (
    pos_cash_missing_summary["missing_count"]
    / len(pos_cash_features)
)


# 3. 查看缺失最多的字段
pos_cash_missing_summary.head(20)

In [ ]:
# 1. 找出比例类字段
pos_ratio_columns = [
    col
    for col in pos_cash_features.columns
    if "RATE" in col or "RATIO" in col
]


# 2. 查看比例特征分布
pos_cash_features[
    pos_ratio_columns
].describe().T

In [ ]:
key_pos_features = [
    "POS_LOAN_COUNT",
    "POS_TOTAL_HISTORY_MONTH_COUNT",
    "POS_TOTAL_DPD_MONTH_COUNT",
    "POS_TOTAL_DPD_MONTH_RATE",
    "POS_MAX_DPD",
    "POS_ACTIVE_LOAN_COUNT",
    "POS_COMPLETED_LOAN_COUNT",
    "POS_TOTAL_FUTURE_INSTALMENT_COUNT",
    "POS_RECENT_MONTH_COUNT",
    "POS_RECENT_DPD_MONTH_COUNT",
    "POS_RECENT_DPD_MONTH_RATE",
    "POS_RECENT_MAX_DPD"
]


for col in key_pos_features:
    print(
        col,
        col in pos_cash_features.columns
    )

In [ ]:
sfrom pathlib import Path


# 1. 建立输出路径
output_path = Path(
    "data/processed/pos_cash_features.parquet"
)


# 2. 保存最终客户级特征表
pos_cash_features.to_parquet(
    output_path,
    index=False
)


# 3. 检查是否保存成功
print(
    "文件是否存在:",
    output_path.exists()
)

print(
    "保存位置:",
    output_path.resolve()
)

print(
    "最终 Shape:",
    pos_cash_features.shape
)